# Advanced Problems with Solutions: Instance, Class, and Static Methods

This notebook develops advanced practice around:

- instance-method binding (`self`)
- class-method binding (`cls`)
- static methods
- class vs. instance state
- inheritance and polymorphic constructors
- method lookup and descriptors
- factories and registries
- validation and utility helpers
- testable time-dependent designs
- refactoring a timer-style class
- API design and best practices

The exercises intentionally go beyond basic syntax. Each problem contains:

1. a precise task,
2. runnable starter/example code,
3. a complete solution,
4. assertions or demonstrations,
5. a short design explanation.

All examples use the Python standard library only.


## 0. Fast Reference

### Instance method

```python
class C:
    def f(self, x):
        ...
```

- Accessed through an instance, it is bound to that instance.
- Use it when behavior needs object-specific state.

### Class method

```python
class C:
    @classmethod
    def f(cls, x):
        ...
```

- Bound to the class, even when accessed through an instance.
- Use it for alternative constructors, class-level configuration, or subclass-aware behavior.

### Static method

```python
class C:
    @staticmethod
    def f(x):
        ...
```

- Receives neither `self` nor `cls` automatically.
- Use it for logic conceptually owned by the class but independent of instance/class state.

### Design rule

Prefer the least powerful method type that correctly expresses the dependency:

- needs instance state → instance method
- needs class/subclass state → class method
- needs neither → static method or module-level function


In [1]:
# Baseline example used throughout the notebook.

class MethodKinds:
    class_value = "shared"

    def instance_method(self):
        return ("instance", self)

    @classmethod
    def class_method(cls):
        return ("class", cls)

    @staticmethod
    def static_method():
        return ("static", None)

obj = MethodKinds()

print(obj.instance_method())
print(MethodKinds.class_method())
print(obj.class_method())
print(MethodKinds.static_method())
print(obj.static_method())


('instance', <__main__.MethodKinds object at 0x0000019AC6124440>)
('class', <class '__main__.MethodKinds'>)
('class', <class '__main__.MethodKinds'>)
('static', None)
('static', None)


# Problem 1 — Predict and Verify Method Binding

Without executing the first section mentally, predict the type and binding behavior of each expression:

```python
BindingLab.regular
BindingLab().regular
BindingLab.classy
BindingLab().classy
BindingLab.static
BindingLab().static
```

Then verify your prediction programmatically.

### Requirements

- Use `inspect.ismethod` and `inspect.isfunction`.
- Inspect the raw class dictionary with `BindingLab.__dict__`.
- Explain why `classmethod` and `staticmethod` appear differently inside the class dictionary.


In [2]:
# Starter code

class BindingLab:
    def regular(self):
        return "regular"

    @classmethod
    def classy(cls):
        return cls.__name__

    @staticmethod
    def static():
        return "static"


## Solution 1

A normal function stored in a class participates in descriptor binding. Accessing it through an instance creates a bound method.

`classmethod` is also a descriptor, but it binds the class rather than the instance.

`staticmethod` suppresses binding and returns the underlying function unchanged.


In [3]:
import inspect

obj = BindingLab()

items = {
    "BindingLab.regular": BindingLab.regular,
    "obj.regular": obj.regular,
    "BindingLab.classy": BindingLab.classy,
    "obj.classy": obj.classy,
    "BindingLab.static": BindingLab.static,
    "obj.static": obj.static,
}

for name, value in items.items():
    print(
        f"{name:22} "
        f"type={type(value).__name__:12} "
        f"ismethod={inspect.ismethod(value)!s:5} "
        f"isfunction={inspect.isfunction(value)!s:5}"
    )

print("\nRaw entries from BindingLab.__dict__:")
for name in ("regular", "classy", "static"):
    raw = BindingLab.__dict__[name]
    print(f"{name:8} -> {raw!r} | raw type={type(raw).__name__}")

assert inspect.isfunction(BindingLab.regular)
assert inspect.ismethod(obj.regular)
assert inspect.ismethod(BindingLab.classy)
assert inspect.ismethod(obj.classy)
assert inspect.isfunction(BindingLab.static)
assert inspect.isfunction(obj.static)


BindingLab.regular     type=function     ismethod=False isfunction=True 
obj.regular            type=method       ismethod=True  isfunction=False
BindingLab.classy      type=method       ismethod=True  isfunction=False
obj.classy             type=method       ismethod=True  isfunction=False
BindingLab.static      type=function     ismethod=False isfunction=True 
obj.static             type=function     ismethod=False isfunction=True 

Raw entries from BindingLab.__dict__:
regular  -> <function BindingLab.regular at 0x0000019AD61ACF40> | raw type=function
classy   -> <classmethod(<function BindingLab.classy at 0x0000019AD61ACFE0>)> | raw type=classmethod
static   -> <staticmethod(<function BindingLab.static at 0x0000019AD61ACEA0>)> | raw type=staticmethod


### Best-practice takeaway

Do not decide between `classmethod` and `staticmethod` based on which call syntax “looks nicer.” Decide based on the data the method genuinely depends on.


# Problem 2 — Build a Subclass-Aware Alternative Constructor

Create a `Temperature` class with:

- an instance attribute storing Celsius,
- `from_fahrenheit(...)`,
- `from_kelvin(...)`,
- `to_fahrenheit()`,
- `to_kelvin()`.

Then create a subclass `LoggedTemperature` that adds a `source` attribute.

The alternative constructors must automatically construct the subclass when called through the subclass.

### Key constraint

Do **not** hard-code `Temperature(...)` inside the alternative constructors.


In [4]:
# Starter API

class Temperature:
    ...

class LoggedTemperature(Temperature):
    ...


## Solution 2

Alternative constructors are one of the strongest use cases for `@classmethod` because `cls(...)` preserves subclass polymorphism.


In [5]:
class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        celsius = float(celsius)
        if celsius < self.ABSOLUTE_ZERO_C:
            raise ValueError("Temperature cannot be below absolute zero.")
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        celsius = (float(fahrenheit) - 32) * 5 / 9
        return cls(celsius)

    @classmethod
    def from_kelvin(cls, kelvin):
        return cls(float(kelvin) - 273.15)

    def to_fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    def to_kelvin(self):
        return self.celsius + 273.15

    def __repr__(self):
        return f"{type(self).__name__}(celsius={self.celsius:.2f})"


class LoggedTemperature(Temperature):
    def __init__(self, celsius, source="unknown"):
        super().__init__(celsius)
        self.source = source

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        obj = super().from_fahrenheit(fahrenheit)
        obj.source = "fahrenheit"
        return obj

    @classmethod
    def from_kelvin(cls, kelvin):
        obj = super().from_kelvin(kelvin)
        obj.source = "kelvin"
        return obj


t1 = Temperature.from_fahrenheit(212)
t2 = LoggedTemperature.from_kelvin(273.15)

print(t1, t1.to_kelvin())
print(t2, t2.source)

assert isinstance(t1, Temperature)
assert type(t1) is Temperature
assert isinstance(t2, LoggedTemperature)
assert abs(t1.celsius - 100) < 1e-12
assert abs(t2.celsius - 0) < 1e-12
assert t2.source == "kelvin"


Temperature(celsius=100.00) 373.15
LoggedTemperature(celsius=0.00) kelvin


### Anti-pattern

```python
@classmethod
def from_kelvin(cls, kelvin):
    return Temperature(kelvin - 273.15)
```

That implementation defeats inheritance. Calling `LoggedTemperature.from_kelvin(...)` would still return a `Temperature`.


# Problem 3 — Class Configuration Without Accidentally Creating Instance State

Implement `ApiClient` with a shared class-level base URL.

Requirements:

- `ApiClient.set_base_url(url)` updates configuration for the class.
- `SpecialApiClient` may maintain its own independent class configuration.
- Existing instances must see subsequent class-level changes.
- Assigning `client.base_url = ...` should be demonstrated as instance shadowing.
- Add `reset_base_url()` that resets only the class on which it is invoked.


## Solution 3


In [6]:
class ApiClient:
    DEFAULT_BASE_URL = "https://api.example.com"
    base_url = DEFAULT_BASE_URL

    @classmethod
    def set_base_url(cls, url):
        url = url.rstrip("/")
        if not url.startswith(("http://", "https://")):
            raise ValueError("base_url must start with http:// or https://")
        cls.base_url = url

    @classmethod
    def reset_base_url(cls):
        cls.base_url = cls.DEFAULT_BASE_URL

    def endpoint(self, path):
        return f"{self.base_url}/{path.lstrip('/')}"


class SpecialApiClient(ApiClient):
    pass


a = ApiClient()
b = ApiClient()
s = SpecialApiClient()

ApiClient.set_base_url("https://v2.example.com/")
print(a.endpoint("/users"))
print(b.endpoint("orders"))

SpecialApiClient.set_base_url("https://special.example.com")
print("ApiClient:", ApiClient.base_url)
print("SpecialApiClient:", SpecialApiClient.base_url)

# Instance shadowing:
a.base_url = "https://local-override.example.com"
print("a:", a.base_url)
print("b:", b.base_url)
print("ApiClient:", ApiClient.base_url)

assert b.base_url == "https://v2.example.com"
assert ApiClient.base_url == "https://v2.example.com"
assert SpecialApiClient.base_url == "https://special.example.com"
assert a.base_url == "https://local-override.example.com"


https://v2.example.com/users
https://v2.example.com/orders
ApiClient: https://v2.example.com
SpecialApiClient: https://special.example.com
a: https://local-override.example.com
b: https://v2.example.com
ApiClient: https://v2.example.com


In [7]:
# Observe exactly where attributes live.

print("a.__dict__ =", a.__dict__)
print("b.__dict__ =", b.__dict__)
print("ApiClient.__dict__['base_url'] =", ApiClient.__dict__["base_url"])
print("SpecialApiClient.__dict__['base_url'] =", SpecialApiClient.__dict__["base_url"])


a.__dict__ = {'base_url': 'https://local-override.example.com'}
b.__dict__ = {}
ApiClient.__dict__['base_url'] = https://v2.example.com
SpecialApiClient.__dict__['base_url'] = https://special.example.com


### Best-practice takeaway

A class attribute accessed through `self` is convenient, but assigning to `self.attribute` creates/updates an instance attribute. When you intend to mutate shared class configuration, write through `cls.attribute`.


# Problem 4 — Static Validation Helper vs Class Method

Design a `Username` class.

Requirements:

- usernames must be strings,
- length must be between configurable class limits,
- allowed characters are letters, digits, `_`, and `-`,
- `normalize(raw)` strips surrounding whitespace and lowercases the input,
- subclasses may change the minimum/maximum lengths.

Decide which operations should be static methods and which should be class methods.

### Goal

Your implementation should allow this subclass to work correctly:

```python
class ShortUsername(Username):
    MIN_LENGTH = 2
    MAX_LENGTH = 8
```


## Solution 4


In [8]:
class Username:
    MIN_LENGTH = 3
    MAX_LENGTH = 20

    def __init__(self, value):
        normalized = self.normalize(value)
        self.validate(normalized)
        self.value = normalized

    @staticmethod
    def normalize(raw):
        if not isinstance(raw, str):
            raise TypeError("username must be a string")
        return raw.strip().lower()

    @staticmethod
    def has_allowed_characters(value):
        return all(ch.isalnum() or ch in "_-" for ch in value)

    @classmethod
    def validate(cls, value):
        if not (cls.MIN_LENGTH <= len(value) <= cls.MAX_LENGTH):
            raise ValueError(
                f"username length must be between "
                f"{cls.MIN_LENGTH} and {cls.MAX_LENGTH}"
            )
        if not cls.has_allowed_characters(value):
            raise ValueError("username contains invalid characters")

    def __repr__(self):
        return f"{type(self).__name__}({self.value!r})"


class ShortUsername(Username):
    MIN_LENGTH = 2
    MAX_LENGTH = 8


u1 = Username("  Alice_42  ")
u2 = ShortUsername("xy")

print(u1)
print(u2)

assert u1.value == "alice_42"
assert u2.value == "xy"

try:
    ShortUsername("this-is-way-too-long")
except ValueError as exc:
    print("Expected:", exc)


Username('alice_42')
ShortUsername('xy')
Expected: username length must be between 2 and 8


### Why this split?

- `normalize` does not depend on class or instance state → `staticmethod`.
- character checking does not depend on class or instance state → `staticmethod`.
- length validation depends on subclass-overridable class constants → `classmethod`.


# Problem 5 — Polymorphic Factory Registry

Build an extensible `Serializer` hierarchy.

Requirements:

- each serializer subclass declares a short format name,
- registration happens through class-level behavior,
- `Serializer.create("json")` returns the registered subclass instance,
- adding a new serializer should not require editing a giant `if/elif` chain,
- duplicate registrations should be rejected,
- include JSON-like and key-value serializers using only the standard library.


## Solution 5


In [9]:
import json

class Serializer:
    _registry = {}

    @classmethod
    def register(cls, name, serializer_cls):
        key = cls._normalize_name(name)
        if key in cls._registry:
            raise ValueError(f"Serializer {key!r} is already registered.")
        if not issubclass(serializer_cls, Serializer):
            raise TypeError("serializer_cls must inherit from Serializer")
        cls._registry[key] = serializer_cls

    @classmethod
    def create(cls, name, **kwargs):
        key = cls._normalize_name(name)
        try:
            serializer_cls = cls._registry[key]
        except KeyError:
            supported = ", ".join(sorted(cls._registry))
            raise ValueError(
                f"Unknown serializer {name!r}. Supported: {supported}"
            ) from None
        return serializer_cls(**kwargs)

    @staticmethod
    def _normalize_name(name):
        if not isinstance(name, str):
            raise TypeError("serializer name must be a string")
        value = name.strip().lower()
        if not value:
            raise ValueError("serializer name cannot be empty")
        return value

    def dumps(self, data):
        raise NotImplementedError


class JsonSerializer(Serializer):
    def __init__(self, *, indent=None):
        self.indent = indent

    def dumps(self, data):
        return json.dumps(data, indent=self.indent, sort_keys=True)


class KeyValueSerializer(Serializer):
    def __init__(self, *, separator="="):
        self.separator = separator

    def dumps(self, data):
        return "\n".join(
            f"{key}{self.separator}{value}"
            for key, value in sorted(data.items())
        )


Serializer.register("json", JsonSerializer)
Serializer.register("kv", KeyValueSerializer)

payload = {"b": 2, "a": 1}

json_s = Serializer.create(" JSON ", indent=2)
kv_s = Serializer.create("kv", separator=":")

print(json_s.dumps(payload))
print("---")
print(kv_s.dumps(payload))

assert isinstance(json_s, JsonSerializer)
assert isinstance(kv_s, KeyValueSerializer)


{
  "a": 1,
  "b": 2
}
---
a:1
b:2


### Extension exercise

Refactor registration so subclasses can auto-register via `__init_subclass__`. The explicit registry version above is often easier to understand and test, while automatic registration can reduce boilerplate in plugin-style systems.


# Problem 6 — Implement Auto-Registration with `__init_subclass__`

Extend the previous idea.

Requirements:

- every concrete subclass declares `format_name`,
- a subclass is automatically added to the registry,
- abstract/intermediate subclasses may opt out,
- duplicate names raise an error,
- `create(...)` remains a class method.


## Solution 6


In [10]:
class AutoSerializer:
    _registry = {}
    format_name = None

    def __init_subclass__(cls, *, register=True, **kwargs):
        super().__init_subclass__(**kwargs)

        if not register:
            return

        if cls.format_name is None:
            raise TypeError(
                f"{cls.__name__} must define format_name "
                f"or use register=False"
            )

        key = cls._normalize_name(cls.format_name)

        if key in AutoSerializer._registry:
            raise ValueError(f"Duplicate serializer name: {key!r}")

        AutoSerializer._registry[key] = cls

    @staticmethod
    def _normalize_name(name):
        return str(name).strip().lower()

    @classmethod
    def create(cls, name, **kwargs):
        key = cls._normalize_name(name)
        try:
            concrete = cls._registry[key]
        except KeyError:
            raise ValueError(f"Unknown format: {name!r}") from None
        return concrete(**kwargs)


class TextSerializer(AutoSerializer, register=False):
    def dumps(self, data):
        raise NotImplementedError


class CsvLikeSerializer(TextSerializer):
    format_name = "csv-like"

    def __init__(self, delimiter=","):
        self.delimiter = delimiter

    def dumps(self, data):
        return self.delimiter.join(map(str, data))


class PipeSerializer(TextSerializer):
    format_name = "pipe"

    def dumps(self, data):
        return "|".join(map(str, data))


s1 = AutoSerializer.create("csv-like", delimiter=";")
s2 = AutoSerializer.create("pipe")

print(s1.dumps([1, 2, 3]))
print(s2.dumps(["a", "b", "c"]))

assert type(s1) is CsvLikeSerializer
assert type(s2) is PipeSerializer


1;2;3
a|b|c


# Problem 7 — Descriptor-Level Introspection

Explain and prove the difference between:

```python
Demo.__dict__["normal"]
Demo.__dict__["classy"]
Demo.__dict__["static"]
```

Then recover the underlying functions from `classmethod` and `staticmethod`.

### Requirements

- inspect `__func__`,
- manually invoke descriptor `__get__`,
- compare manual binding with normal attribute access.


## Solution 7


In [11]:
class Demo:
    def normal(self, x):
        return ("normal", self, x)

    @classmethod
    def classy(cls, x):
        return ("classy", cls, x)

    @staticmethod
    def static(x):
        return ("static", x)


normal_raw = Demo.__dict__["normal"]
classy_raw = Demo.__dict__["classy"]
static_raw = Demo.__dict__["static"]

print("raw normal:", normal_raw)
print("raw classy:", classy_raw)
print("raw static:", static_raw)

print("\nUnderlying functions:")
print("classmethod.__func__:", classy_raw.__func__)
print("staticmethod.__func__:", static_raw.__func__)

obj = Demo()

manual_normal = normal_raw.__get__(obj, Demo)
manual_classy = classy_raw.__get__(obj, Demo)
manual_static = static_raw.__get__(obj, Demo)

print("\nManual binding:")
print(manual_normal("A"))
print(manual_classy("B"))
print(manual_static("C"))

assert manual_normal.__self__ is obj
assert manual_classy.__self__ is Demo
assert manual_static is Demo.static


raw normal: <function Demo.normal at 0x0000019AD61AE980>
raw classy: <classmethod(<function Demo.classy at 0x0000019AD61AE8E0>)>
raw static: <staticmethod(<function Demo.static at 0x0000019AD61AEA20>)>

Underlying functions:
classmethod.__func__: <function Demo.classy at 0x0000019AD61AE8E0>
staticmethod.__func__: <function Demo.static at 0x0000019AD61AEA20>

Manual binding:
('normal', <__main__.Demo object at 0x0000019AD610F620>, 'A')
('classy', <class '__main__.Demo'>, 'B')
('static', 'C')


### Important insight

Decorators such as `@classmethod` and `@staticmethod` do not merely “mark” functions. They replace the function stored in the class namespace with descriptor objects that control attribute access behavior.


# Problem 8 — Refactor a Bad Utility Class

The following design overuses static methods:

```python
class PriceTools:
    tax_rate = 0.20

    @staticmethod
    def normalize(amount): ...
    @staticmethod
    def tax(amount): ...
    @staticmethod
    def total(amount): ...
```

Problem: `tax()` and `total()` depend on class configuration, yet `staticmethod` gives them no `cls`.

Refactor the class so subclasses can override `tax_rate` and automatically get correct results.


## Solution 8


In [12]:
from decimal import Decimal, ROUND_HALF_UP

class PriceTools:
    tax_rate = Decimal("0.20")

    @staticmethod
    def normalize(amount):
        value = Decimal(str(amount))
        if value < 0:
            raise ValueError("amount cannot be negative")
        return value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

    @classmethod
    def tax(cls, amount):
        amount = cls.normalize(amount)
        return (amount * cls.tax_rate).quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )

    @classmethod
    def total(cls, amount):
        amount = cls.normalize(amount)
        return amount + cls.tax(amount)


class ReducedTaxPriceTools(PriceTools):
    tax_rate = Decimal("0.05")


print("standard:", PriceTools.total("100"))
print("reduced :", ReducedTaxPriceTools.total("100"))

assert PriceTools.total("100") == Decimal("120.00")
assert ReducedTaxPriceTools.total("100") == Decimal("105.00")


standard: 120.00
reduced : 105.00


### Best-practice takeaway

If a method uses a class attribute that subclasses are expected to override, a class method is usually the correct abstraction.


# Problem 9 — Immutable Value Object with Multiple Constructors

Create an immutable `Point` value object that supports:

- `Point(x, y)`,
- `Point.from_string("10.5, -2")`,
- `Point.from_polar(radius, angle_radians)`,
- `distance_to(other)` as an instance method,
- `is_finite_number(value)` as a static utility.

Use `dataclasses.dataclass(frozen=True)`.


## Solution 9


In [13]:
from dataclasses import dataclass
from math import cos, sin, hypot, isfinite

@dataclass(frozen=True)
class Point:
    x: float
    y: float

    def __post_init__(self):
        if not self.is_finite_number(self.x):
            raise ValueError("x must be finite")
        if not self.is_finite_number(self.y):
            raise ValueError("y must be finite")

    @staticmethod
    def is_finite_number(value):
        try:
            return isfinite(float(value))
        except (TypeError, ValueError):
            return False

    @classmethod
    def from_string(cls, text):
        left, right = text.split(",", maxsplit=1)
        return cls(float(left.strip()), float(right.strip()))

    @classmethod
    def from_polar(cls, radius, angle_radians):
        radius = float(radius)
        angle_radians = float(angle_radians)
        if radius < 0:
            raise ValueError("radius cannot be negative")
        return cls(
            radius * cos(angle_radians),
            radius * sin(angle_radians)
        )

    def distance_to(self, other):
        if not isinstance(other, Point):
            raise TypeError("other must be a Point")
        return hypot(self.x - other.x, self.y - other.y)


p1 = Point.from_string("3, 4")
p2 = Point.from_polar(5, 0)

print("p1 =", p1)
print("p2 =", p2)
print("distance =", p1.distance_to(p2))

assert p1 == Point(3.0, 4.0)
assert abs(p2.x - 5.0) < 1e-12
assert abs(p2.y) < 1e-12


p1 = Point(x=3.0, y=4.0)
p2 = Point(x=5.0, y=0.0)
distance = 4.47213595499958


# Problem 10 — Design a Testable Clock Instead of Hard-Coding `datetime.now`

A timer implementation that directly calls `datetime.now(...)` is difficult to test deterministically.

Refactor the idea into a `Timer` that accepts an injectable clock.

Requirements:

- default clock returns aware UTC datetimes,
- `start()` and `stop()` are instance methods,
- class-level timezone affects display only,
- `set_timezone(...)` is a class method,
- duration calculation stays in UTC,
- tests must not use `sleep`,
- stopping before starting raises a custom exception.


## Solution 10

This is a more testable design than making every test depend on real wall-clock timing.


In [14]:
from datetime import datetime, timezone, timedelta

class TimerError(RuntimeError):
    pass


class Timer:
    tz = timezone.utc

    def __init__(self, *, clock=None):
        self._clock = clock or self.system_utc_now
        self._started_at_utc = None
        self._stopped_at_utc = None

    @staticmethod
    def system_utc_now():
        return datetime.now(timezone.utc)

    @classmethod
    def set_timezone(cls, *, offset_hours=0, name="UTC"):
        cls.tz = timezone(timedelta(hours=offset_hours), name)

    def start(self):
        self._started_at_utc = self._require_aware_utc(self._clock())
        self._stopped_at_utc = None
        return self

    def stop(self):
        if self._started_at_utc is None:
            raise TimerError("Timer must be started before it can be stopped.")
        self._stopped_at_utc = self._require_aware_utc(self._clock())
        return self

    @staticmethod
    def _require_aware_utc(value):
        if not isinstance(value, datetime):
            raise TypeError("clock must return datetime objects")
        if value.tzinfo is None:
            raise ValueError("clock must return timezone-aware datetimes")
        return value.astimezone(timezone.utc)

    @property
    def start_time(self):
        if self._started_at_utc is None:
            raise TimerError("Timer has not been started.")
        return self._started_at_utc.astimezone(type(self).tz)

    @property
    def end_time(self):
        if self._stopped_at_utc is None:
            raise TimerError("Timer has not been stopped.")
        return self._stopped_at_utc.astimezone(type(self).tz)

    @property
    def elapsed_seconds(self):
        if self._started_at_utc is None:
            raise TimerError("Timer has not been started.")

        end = (
            self._stopped_at_utc
            if self._stopped_at_utc is not None
            else self._require_aware_utc(self._clock())
        )
        return (end - self._started_at_utc).total_seconds()


In [15]:
# Deterministic fake clock: no sleep() required.

class FakeClock:
    def __init__(self, *times):
        self._times = iter(times)

    def __call__(self):
        return next(self._times)


t0 = datetime(2030, 1, 1, 12, 0, tzinfo=timezone.utc)
t1 = t0 + timedelta(seconds=2.5)

timer = Timer(clock=FakeClock(t0, t1))
timer.start()
timer.stop()

print("elapsed:", timer.elapsed_seconds)
assert timer.elapsed_seconds == 2.5

Timer.set_timezone(offset_hours=-5, name="UTC-5")
print("display start:", timer.start_time)
print("display end  :", timer.end_time)

assert timer.start_time.utcoffset() == timedelta(hours=-5)


elapsed: 2.5
display start: 2030-01-01 07:00:00-05:00
display end  : 2030-01-01 07:00:02.500000-05:00


### Best-practice takeaway

Dependency injection is often more important than whether a helper is technically a static method. The static default clock is convenient, but the instance accepts an alternate callable for deterministic tests.


# Problem 11 — Subclass-Specific Timer Configuration

Create two timer subclasses with independent display timezones:

```python
class NewYorkTimer(Timer): ...
class TokyoTimer(Timer): ...
```

Demonstrate that changing one subclass's class configuration does not alter the other subclass or the base class.


## Solution 11


In [16]:
class NewYorkTimer(Timer):
    pass


class TokyoTimer(Timer):
    pass


# Fixed offsets are used here for the exercise.
# Real civil-time applications should account for daylight-saving rules.
NewYorkTimer.set_timezone(offset_hours=-5, name="NY-fixed")
TokyoTimer.set_timezone(offset_hours=9, name="Tokyo-fixed")

print("Timer:", Timer.tz)
print("NY   :", NewYorkTimer.tz)
print("Tokyo:", TokyoTimer.tz)

assert NewYorkTimer.tz is not TokyoTimer.tz
assert NewYorkTimer.tz.utcoffset(None) == timedelta(hours=-5)
assert TokyoTimer.tz.utcoffset(None) == timedelta(hours=9)


Timer: UTC-5
NY   : NY-fixed
Tokyo: Tokyo-fixed


# Problem 12 — Method Type Code Review

For each method below, decide whether it should be:

- instance method,
- class method,
- static method,
- or a module-level function.

```python
class Order:
    tax_rate = 0.2

    def subtotal(...): ...
    def from_json(...): ...
    def validate_sku(...): ...
    def total(...): ...
    def supported_countries(...): ...
```

Assume:

- `subtotal` uses the order's line items,
- `from_json` constructs an order or subclass,
- `validate_sku` checks only formatting rules,
- `total` uses both line items and `tax_rate`,
- `supported_countries` returns a constant tuple that is conceptually global rather than class-specific.


## Solution 12

Recommended design:

- `subtotal` → instance method
- `from_json` → class method
- `validate_sku` → static method if it stays conceptually attached to `Order`
- `total` → instance method because it uses instance data; read the class policy through `type(self).tax_rate`
- `supported_countries` → module-level constant/function if it is truly global


In [17]:
import json

SUPPORTED_COUNTRIES = ("BG", "DE", "FR", "US")


class Order:
    tax_rate = 0.20

    def __init__(self, lines):
        self.lines = list(lines)

    @staticmethod
    def validate_sku(sku):
        if not isinstance(sku, str):
            raise TypeError("sku must be a string")
        return sku.isalnum() and 3 <= len(sku) <= 20

    @classmethod
    def from_json(cls, text):
        data = json.loads(text)
        return cls(data["lines"])

    def subtotal(self):
        return sum(line["qty"] * line["unit_price"] for line in self.lines)

    def total(self):
        return self.subtotal() * (1 + type(self).tax_rate)


class LowTaxOrder(Order):
    tax_rate = 0.05


raw = json.dumps({
    "lines": [
        {"sku": "ABC123", "qty": 2, "unit_price": 10.0},
        {"sku": "XYZ999", "qty": 1, "unit_price": 5.0},
    ]
})

order = LowTaxOrder.from_json(raw)

print("subtotal:", order.subtotal())
print("total   :", order.total())
print("countries:", SUPPORTED_COUNTRIES)

assert isinstance(order, LowTaxOrder)
assert order.subtotal() == 25.0
assert order.total() == 26.25


subtotal: 25.0
total   : 26.25
countries: ('BG', 'DE', 'FR', 'US')


# Problem 13 — Build a Parsing Hierarchy with Covariant Constructors

Create a base `Record` class and two subclasses:

- `UserRecord`
- `ProductRecord`

Each subclass should support `from_mapping(...)`.

Then add `from_pairs(...)` to the base class in a way that delegates to the subclass-aware constructor.

### Requirements

Calling:

```python
UserRecord.from_pairs(...)
ProductRecord.from_pairs(...)
```

must return the correct subclass without duplicating parsing logic.


## Solution 13


In [18]:
class Record:
    @classmethod
    def from_mapping(cls, mapping):
        return cls(**mapping)

    @classmethod
    def from_pairs(cls, pairs):
        return cls.from_mapping(dict(pairs))


class UserRecord(Record):
    def __init__(self, username, active):
        self.username = username
        self.active = bool(active)

    def __repr__(self):
        return f"UserRecord(username={self.username!r}, active={self.active})"


class ProductRecord(Record):
    def __init__(self, sku, price):
        self.sku = sku
        self.price = float(price)

    def __repr__(self):
        return f"ProductRecord(sku={self.sku!r}, price={self.price})"


u = UserRecord.from_pairs([("username", "alice"), ("active", 1)])
p = ProductRecord.from_pairs([("sku", "P100"), ("price", "19.99")])

print(u)
print(p)

assert type(u) is UserRecord
assert type(p) is ProductRecord


UserRecord(username='alice', active=True)
ProductRecord(sku='P100', price=19.99)


### Design insight

`cls.from_mapping(...)` is usually preferable to calling `Record.from_mapping(...)` because it preserves the active subclass throughout the construction pipeline.


# Problem 14 — Class Method Inheritance and `super()`

Predict what `cls` is at every stage of the following call:

```python
Child.describe()
```

Then implement the hierarchy and verify it.

The child should extend the parent output without hard-coding class names.


## Solution 14


In [19]:
class Parent:
    kind = "parent"

    @classmethod
    def describe(cls):
        return {
            "runtime_cls": cls.__name__,
            "kind": cls.kind,
        }


class Child(Parent):
    kind = "child"

    @classmethod
    def describe(cls):
        data = super().describe()
        data["extended_by"] = cls.__name__
        return data


print(Parent.describe())
print(Child.describe())

assert Parent.describe() == {
    "runtime_cls": "Parent",
    "kind": "parent",
}

assert Child.describe() == {
    "runtime_cls": "Child",
    "kind": "child",
    "extended_by": "Child",
}


{'runtime_cls': 'Parent', 'kind': 'parent'}
{'runtime_cls': 'Child', 'kind': 'child', 'extended_by': 'Child'}


### Key point

Even when `Parent.describe` is reached through `super()`, the bound class remains the runtime subclass (`Child`). This is what makes class methods naturally polymorphic.


# Problem 15 — Static Method Inheritance: What Does *Not* Happen?

Create a static method that refers directly to the base class and show why that may be surprising in subclasses.

Then refactor the behavior into a class method.

### Scenario

A label generator should include the active subclass name.


## Solution 15


In [20]:
class BadLabel:
    @staticmethod
    def label():
        # Hard-coded base class reference:
        return f"label:{BadLabel.__name__}"


class BadChildLabel(BadLabel):
    pass


print(BadLabel.label())
print(BadChildLabel.label())

# Both are the same, because no cls is available.
assert BadChildLabel.label() == "label:BadLabel"


class GoodLabel:
    @classmethod
    def label(cls):
        return f"label:{cls.__name__}"


class GoodChildLabel(GoodLabel):
    pass


print(GoodLabel.label())
print(GoodChildLabel.label())

assert GoodChildLabel.label() == "label:GoodChildLabel"


label:BadLabel
label:BadLabel
label:GoodLabel
label:GoodChildLabel


# Problem 16 — Advanced Factory with Validation Pipeline

Design an `Account` hierarchy where:

- `Account.from_dict(data)` validates and constructs instances,
- common validation lives in the base class,
- subclasses can add validation,
- `cls(...)` must produce the subclass,
- helper functions that need no class state should remain static methods.

Implement `BusinessAccount` requiring a non-empty company name.


## Solution 16


In [21]:
class Account:
    def __init__(self, username, email):
        self.username = username
        self.email = email

    @staticmethod
    def _require_nonempty_string(value, field):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{field} must be a non-empty string")
        return value.strip()

    @classmethod
    def _validate_dict(cls, data):
        if not isinstance(data, dict):
            raise TypeError("data must be a dict")

        username = cls._require_nonempty_string(
            data.get("username"), "username"
        )
        email = cls._require_nonempty_string(
            data.get("email"), "email"
        )

        if "@" not in email:
            raise ValueError("email must contain '@'")

        return {
            "username": username,
            "email": email,
        }

    @classmethod
    def from_dict(cls, data):
        validated = cls._validate_dict(data)
        return cls(**validated)


class BusinessAccount(Account):
    def __init__(self, username, email, company):
        super().__init__(username, email)
        self.company = company

    @classmethod
    def _validate_dict(cls, data):
        validated = super()._validate_dict(data)
        company = cls._require_nonempty_string(
            data.get("company"), "company"
        )
        validated["company"] = company
        return validated


a = Account.from_dict({
    "username": "alice",
    "email": "alice@example.com",
})

b = BusinessAccount.from_dict({
    "username": "bob",
    "email": "bob@example.com",
    "company": "Acme",
})

print(type(a).__name__, a.__dict__)
print(type(b).__name__, b.__dict__)

assert type(a) is Account
assert type(b) is BusinessAccount


Account {'username': 'alice', 'email': 'alice@example.com'}
BusinessAccount {'username': 'bob', 'email': 'bob@example.com', 'company': 'Acme'}


# Problem 17 — Testing Method Semantics Explicitly

Write tests that prove:

1. instance methods bind to instances,
2. class methods bind to classes,
3. static methods remain plain functions,
4. subclass class-method binding targets the subclass,
5. the same static function object can be retrieved through class and instance access.


## Solution 17


In [22]:
import inspect

class Semantics:
    def instance(self):
        return type(self)

    @classmethod
    def klass(cls):
        return cls

    @staticmethod
    def utility(x):
        return x * 2


class SemanticsChild(Semantics):
    pass


obj = Semantics()
child = SemanticsChild()

# 1
assert inspect.ismethod(obj.instance)
assert obj.instance.__self__ is obj

# 2
assert inspect.ismethod(Semantics.klass)
assert Semantics.klass.__self__ is Semantics

# 3
assert inspect.isfunction(Semantics.utility)
assert inspect.isfunction(obj.utility)

# 4
assert SemanticsChild.klass.__self__ is SemanticsChild
assert child.klass.__self__ is SemanticsChild

# 5
assert Semantics.utility is obj.utility

print("All binding-semantics assertions passed.")


All binding-semantics assertions passed.


# Problem 18 — Refactor for Clearer API Ownership

Suppose a class contains:

```python
class Report:
    @staticmethod
    def parse_date(...): ...
    @staticmethod
    def slugify(...): ...
    @staticmethod
    def calculate_checksum(...): ...
    @staticmethod
    def open_database_connection(...): ...
```

Which helpers belong in the class, and which are probably better as module-level functions or separate services?

### Solution discussion

Method type is only part of API design. A static method can be technically valid while still being poorly placed.

A useful heuristic:

- keep a helper as `staticmethod` when it is tightly coupled to the domain concept and helps discoverability,
- move broadly reusable helpers to a module,
- move infrastructure concerns (database/network/filesystem) into separate collaborators when they are not intrinsic to the domain object,
- avoid turning a class into a namespace for unrelated utilities.

The goal is cohesive ownership, not maximizing use of decorators.


In [23]:
# Example of a cohesive static helper.

import re

class Report:
    def __init__(self, title):
        self.title = title

    @staticmethod
    def slugify_title(title):
        normalized = title.strip().lower()
        normalized = re.sub(r"[^a-z0-9]+", "-", normalized)
        return normalized.strip("-")

    @property
    def slug(self):
        return self.slugify_title(self.title)


r = Report("  Quarterly Revenue: Q4  ")
print(r.slug)

assert r.slug == "quarterly-revenue-q4"


quarterly-revenue-q4


# Capstone Problem — Production-Style Timer API

Build a robust timer abstraction combining the main ideas from this notebook.

### Requirements

The timer must support:

- class-level display timezone,
- subclass-specific timezone overrides,
- dependency-injected clocks,
- start/stop/reset,
- context-manager usage,
- elapsed duration as `timedelta`,
- human-readable elapsed formatting,
- alternative constructor `started(...)`,
- validation of aware datetimes,
- custom exceptions,
- no `sleep()` in tests.

### Design requirements

Use:

- instance methods for per-timer state,
- class methods for subclass-aware construction/configuration,
- static methods for pure helpers.


## Capstone Solution


In [24]:
from datetime import datetime, timezone, timedelta

class AdvancedTimerError(RuntimeError):
    pass


class AdvancedTimer:
    display_tz = timezone.utc

    def __init__(self, *, clock=None):
        self._clock = clock or self.system_utc_now
        self._start_utc = None
        self._end_utc = None

    # ---------- static methods: pure/local utilities ----------

    @staticmethod
    def system_utc_now():
        return datetime.now(timezone.utc)

    @staticmethod
    def ensure_aware(value):
        if not isinstance(value, datetime):
            raise TypeError("clock must return datetime")
        if value.tzinfo is None or value.utcoffset() is None:
            raise ValueError("datetime must be timezone-aware")
        return value

    @staticmethod
    def format_duration(duration):
        if not isinstance(duration, timedelta):
            raise TypeError("duration must be timedelta")

        total = duration.total_seconds()
        sign = "-" if total < 0 else ""
        total = abs(total)

        hours, remainder = divmod(total, 3600)
        minutes, seconds = divmod(remainder, 60)

        return (
            f"{sign}{int(hours):02d}:"
            f"{int(minutes):02d}:"
            f"{seconds:06.3f}"
        )

    # ---------- class methods: class/subclass policy ----------

    @classmethod
    def set_display_timezone(cls, *, offset_hours=0, name="UTC"):
        cls.display_tz = timezone(
            timedelta(hours=offset_hours),
            name
        )

    @classmethod
    def started(cls, *, clock=None):
        return cls(clock=clock).start()

    # ---------- instance methods: per-object state ----------

    def _now_utc(self):
        return self.ensure_aware(self._clock()).astimezone(timezone.utc)

    def start(self):
        self._start_utc = self._now_utc()
        self._end_utc = None
        return self

    def stop(self):
        if self._start_utc is None:
            raise AdvancedTimerError(
                "Cannot stop a timer that has not been started."
            )
        self._end_utc = self._now_utc()
        return self

    def reset(self):
        self._start_utc = None
        self._end_utc = None
        return self

    @property
    def is_running(self):
        return self._start_utc is not None and self._end_utc is None

    @property
    def start_time(self):
        if self._start_utc is None:
            raise AdvancedTimerError("Timer has not been started.")
        return self._start_utc.astimezone(type(self).display_tz)

    @property
    def end_time(self):
        if self._end_utc is None:
            raise AdvancedTimerError("Timer has not been stopped.")
        return self._end_utc.astimezone(type(self).display_tz)

    @property
    def elapsed(self):
        if self._start_utc is None:
            raise AdvancedTimerError("Timer has not been started.")

        end = self._end_utc if self._end_utc is not None else self._now_utc()
        return end - self._start_utc

    @property
    def elapsed_text(self):
        return self.format_duration(self.elapsed)

    def __enter__(self):
        return self.start()

    def __exit__(self, exc_type, exc, tb):
        self.stop()
        return False


In [25]:
# Capstone deterministic test helpers

class SequenceClock:
    def __init__(self, times):
        self._times = iter(times)

    def __call__(self):
        return next(self._times)


base = datetime(2040, 1, 1, 0, 0, tzinfo=timezone.utc)

clock = SequenceClock([
    base,
    base + timedelta(seconds=65.25),
])

timer = AdvancedTimer.started(clock=clock)
assert timer.is_running

timer.stop()

print("elapsed =", timer.elapsed)
print("text    =", timer.elapsed_text)

assert timer.elapsed == timedelta(seconds=65.25)
assert timer.elapsed_text == "00:01:05.250"
assert not timer.is_running


elapsed = 0:01:05.250000
text    = 00:01:05.250


In [26]:
# Subclass-specific timezone behavior

class SofiaTimer(AdvancedTimer):
    pass


class FixedOffsetTimer(AdvancedTimer):
    pass


SofiaTimer.set_display_timezone(offset_hours=3, name="UTC+3")
FixedOffsetTimer.set_display_timezone(offset_hours=-7, name="UTC-7")

clock1 = SequenceClock([
    base,
    base + timedelta(seconds=1),
])
clock2 = SequenceClock([
    base,
    base + timedelta(seconds=1),
])

sofia = SofiaTimer.started(clock=clock1).stop()
west = FixedOffsetTimer.started(clock=clock2).stop()

print("Sofia start:", sofia.start_time)
print("West start :", west.start_time)

assert sofia.start_time.utcoffset() == timedelta(hours=3)
assert west.start_time.utcoffset() == timedelta(hours=-7)
assert AdvancedTimer.display_tz == timezone.utc


Sofia start: 2040-01-01 03:00:00+03:00
West start : 2039-12-31 17:00:00-07:00


In [27]:
# Context manager test without sleeping.

ctx_clock = SequenceClock([
    base,
    base + timedelta(milliseconds=750),
])

with AdvancedTimer(clock=ctx_clock) as measured:
    # Work would happen here.
    pass

print(measured.elapsed_text)
assert measured.elapsed == timedelta(milliseconds=750)
assert measured.elapsed_text == "00:00:00.750"


00:00:00.750


# Additional Mini-Problems

Use these for extra practice before looking at the compact solutions below.

### A. Which decorator?

A method must return a subclass-specific cache key using `cls.__name__`.

### B. Which decorator?

A helper converts bytes to a lowercase hexadecimal string and uses no class/instance state.

### C. Which decorator?

A method updates an object's `_status` field.

### D. Bug hunt

Why is this usually wrong?

```python
class Base:
    @classmethod
    def make(cls):
        return Base()
```

### E. Shadowing

What changes after:

```python
obj.limit = 10
```

if `limit` previously existed only as a class attribute?

### F. Call syntax

Can a static method be called from an instance? If so, does that make it an instance method?

### Compact solutions

A. `@classmethod`  
B. `@staticmethod` or module-level function  
C. instance method  
D. hard-coding `Base()` breaks subclass-aware construction; use `cls()`  
E. the instance now has its own `limit` attribute that shadows the class attribute  
F. yes; no automatic binding occurs, so it remains static behavior


In [28]:
# Mini-problem demonstrations

class Base:
    limit = 100

    @classmethod
    def cache_key(cls):
        return f"cache:{cls.__name__}"

    @staticmethod
    def bytes_to_hex(data):
        return bytes(data).hex()

    def set_status(self, status):
        self._status = status


class Child(Base):
    pass


obj = Child()

print(obj.cache_key())
print(obj.bytes_to_hex(b"\xCA\xFE"))

print("before shadowing:", obj.limit, obj.__dict__)
obj.limit = 10
print("after shadowing :", obj.limit, obj.__dict__)
print("class unchanged :", Child.limit)

assert obj.cache_key() == "cache:Child"
assert obj.bytes_to_hex(b"\xCA\xFE") == "cafe"
assert obj.limit == 10
assert Child.limit == 100


cache:Child
cafe
before shadowing: 100 {}
after shadowing : 10 {'limit': 10}
class unchanged : 100


# Best-Practice Checklist

Before choosing a method type, ask:

1. Does this operation need object-specific state?
   - yes → instance method

2. Does it need the runtime class or subclass?
   - yes → class method

3. Is it an alternative constructor?
   - usually → class method returning `cls(...)`

4. Is the logic pure and independent of class/instance state?
   - static method if it is strongly related to the class
   - otherwise consider a module-level function

5. Should subclasses override configuration?
   - prefer `cls.attribute` or `type(self).attribute` when polymorphism matters

6. Are you mutating class state?
   - write through `cls`, not `self`

7. Are you hard-coding the base class in a constructor/factory?
   - usually replace `Base(...)` with `cls(...)`

8. Are time, random values, network access, or external resources involved?
   - inject dependencies so tests remain deterministic

9. Are static methods accumulating unrelated utilities?
   - reconsider class cohesion

10. Can assertions prove the intended binding semantics?
    - add tests for `__self__`, subclass construction, state sharing, and shadowing


# Final Challenge — Write Your Own

Create a `Document` hierarchy with:

- `Document.from_text(...)`
- `Document.from_file(...)`
- subclass-specific default encoding
- a static helper for normalizing line endings
- instance methods for word count and summary statistics
- a registry that maps file extensions to subclasses
- deterministic tests using temporary files
- at least one subclass override that proves `cls` is being propagated correctly

Suggested subclasses:

- `MarkdownDocument`
- `PlainTextDocument`
- `LogDocument`

Try solving it without referring to earlier solutions, then compare your design choices against the checklist above.


In [29]:
# Optional starter skeleton for the final challenge.

class Document:
    default_encoding = "utf-8"
    _registry = {}

    def __init__(self, text):
        self.text = text

    @staticmethod
    def normalize_newlines(text):
        return text.replace("\r\n", "\n").replace("\r", "\n")

    @classmethod
    def from_text(cls, text):
        normalized = cls.normalize_newlines(text)
        return cls(normalized)

    @classmethod
    def register_extension(cls, extension, document_cls):
        cls._registry[extension.lower()] = document_cls

    def word_count(self):
        return len(self.text.split())


# Continue from here:
# - from_file
# - extension lookup
# - subclasses
# - validation
# - tests
